# 01 — MovieLens Dataset Audit

**Project:** TASTERA  
**Stage:** Data understanding and quality assessment  
**Dataset:** MovieLens Latest Small (ml-latest-small)  
**Status:** Run locally to generate verified audit results.

## 1. Objective
Inspect source files before preprocessing or modeling. Check schemas, dimensions, missing values, duplicate records, identifier consistency, rating behavior, user activity, movie popularity, genre coverage, and tag coverage.

## 2. Reproducibility and data handling
- Keep original downloaded CSV files unchanged under data/raw/.
- This notebook reads source data and creates plots/tables in notebook output only; it does not rewrite source CSVs.
- Run cells from top to bottom. Record the Python/package environment when reporting results.
- Review the included dataset README and current GroupLens terms before redistribution or commercial use.

Expected local data directory: data/raw/movielens/ml-latest-small/.

## 3. Imports and paths
Load analysis libraries and locate the dataset whether Jupyter starts in the repository root or notebooks/.

In [ ]:
from pathlib import Path
import platform
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 100)

candidate_paths = [
    Path('data/raw/movielens/ml-latest-small'),
    Path('../data/raw/movielens/ml-latest-small'),
    Path.cwd() / 'data/raw/movielens/ml-latest-small',
    Path.cwd().parent / 'data/raw/movielens/ml-latest-small',
]
DATA_DIR = next((p.resolve() for p in candidate_paths if (p / 'movies.csv').exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError('Could not find movies.csv. Place files in data/raw/movielens/ml-latest-small/ or update DATA_DIR.')

print('Python:', sys.version.split()[0])
print('Platform:', platform.platform())
print('pandas:', pd.__version__)
print('NumPy:', np.__version__)
print('Dataset directory:', DATA_DIR)

## 4. Load source files
Load the four CSVs separately and inspect dimensions, columns, data types, and sample records.

In [ ]:
file_names = ['movies.csv', 'ratings.csv', 'tags.csv', 'links.csv']
missing_files = [name for name in file_names if not (DATA_DIR / name).exists()]
if missing_files:
    raise FileNotFoundError(f'Missing expected dataset files: {missing_files}')

movies = pd.read_csv(DATA_DIR / 'movies.csv')
ratings = pd.read_csv(DATA_DIR / 'ratings.csv')
tags = pd.read_csv(DATA_DIR / 'tags.csv')
links = pd.read_csv(DATA_DIR / 'links.csv')
datasets = {'movies': movies, 'ratings': ratings, 'tags': tags, 'links': links}

schema_summary = pd.DataFrame([
    {'dataset': name, 'rows': df.shape[0], 'columns': df.shape[1], 'column_names': ', '.join(df.columns)}
    for name, df in datasets.items()
])
display(schema_summary)

for name, df in datasets.items():
    print(f'\n{name.upper()} — first five rows')
    display(df.head())
    print('Data types:')
    display(df.dtypes.rename('dtype').to_frame())

## 5. Missing values and duplicate rows
Report missing values and duplicate full rows before deciding whether cleaning is appropriate.

In [ ]:
missing_rows = []
for name, df in datasets.items():
    for column in df.columns:
        count = int(df[column].isna().sum())
        missing_rows.append({'dataset': name, 'column': column, 'missing_count': count,
                             'missing_percent': 100 * count / len(df) if len(df) else np.nan})
display(pd.DataFrame(missing_rows))

duplicate_summary = pd.DataFrame([
    {'dataset': name, 'duplicate_full_rows': int(df.duplicated().sum()), 'rows': len(df)}
    for name, df in datasets.items()
])
display(duplicate_summary)

## 6. Identifier consistency
Check expected key uniqueness and whether ratings, tags, and links refer to movie IDs in the catalog. These checks are diagnostic; no records are changed automatically.

In [ ]:
movie_ids = set(movies['movieId'].dropna().astype(int))
link_ids = set(links['movieId'].dropna().astype(int))
rating_movie_ids = set(ratings['movieId'].dropna().astype(int))
tag_movie_ids = set(tags['movieId'].dropna().astype(int))

id_checks = pd.DataFrame([
    {'check': 'movies.movieId unique', 'value': bool(movies['movieId'].is_unique)},
    {'check': 'links.movieId unique', 'value': bool(links['movieId'].is_unique)},
    {'check': 'movie IDs missing from links', 'value': len(movie_ids - link_ids)},
    {'check': 'link IDs absent from movies', 'value': len(link_ids - movie_ids)},
    {'check': 'rating movie IDs absent from movies', 'value': len(rating_movie_ids - movie_ids)},
    {'check': 'tag movie IDs absent from movies', 'value': len(tag_movie_ids - movie_ids)},
    {'check': 'ratings userId null count', 'value': int(ratings['userId'].isna().sum())},
    {'check': 'ratings movieId null count', 'value': int(ratings['movieId'].isna().sum())},
])
display(id_checks)
print('Movie IDs in catalog:', len(movie_ids))
print('Movie IDs in links:', len(link_ids))
print('Movie IDs appearing in ratings:', len(rating_movie_ids))
print('Movie IDs appearing in tags:', len(tag_movie_ids))

## 7. Rating distribution and activity
Inspect the rating scale, overall distribution, and how ratings are distributed across users and movies. Ratings are observed feedback, not a complete record of taste.

In [ ]:
print('Rating summary:')
display(ratings['rating'].describe().to_frame().T)
print('Observed rating values and counts:')
display(ratings['rating'].value_counts(dropna=False).sort_index().rename_axis('rating').to_frame('count'))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
ratings['rating'].value_counts().sort_index().plot(kind='bar', ax=axes[0])
axes[0].set_title('Rating counts')
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Number of ratings')
axes[0].tick_params(axis='x', rotation=0)

ratings_per_user = ratings.groupby('userId').size()
ratings_per_movie = ratings.groupby('movieId').size()
ratings_per_user.plot(kind='hist', bins=30, ax=axes[1])
axes[1].set_title('Ratings per user')
axes[1].set_xlabel('Number of ratings')
axes[1].set_ylabel('Number of users')
plt.tight_layout()
plt.show()

display(pd.DataFrame({'ratings_per_user': ratings_per_user.describe(),
                      'ratings_per_movie': ratings_per_movie.describe()}))

## 8. Rating sparsity
A user–movie matrix is sparse because each user rates only a subset of the catalog. Sparsity affects collaborative filtering and cold-start behavior.

In [ ]:
n_users = ratings['userId'].nunique()
n_rated_movies = ratings['movieId'].nunique()
n_observed = len(ratings)
possible_pairs = n_users * n_rated_movies
density = n_observed / possible_pairs if possible_pairs else np.nan
sparsity = 1 - density if possible_pairs else np.nan

sparsity_summary = pd.DataFrame([{
    'unique_users': n_users,
    'unique_rated_movies': n_rated_movies,
    'observed_ratings': n_observed,
    'possible_user_movie_pairs_for_rated_catalog': possible_pairs,
    'observed_density_percent': density * 100,
    'sparsity_percent': sparsity * 100,
}])
display(sparsity_summary)

## 9. Genre coverage and frequency
Genres are pipe-separated labels. Split them for descriptive analysis while leaving movies.csv unchanged.

In [ ]:
genre_counts = (movies['genres'].fillna('').str.split('|').explode().str.strip()
                .replace('', np.nan).dropna().value_counts())
display(genre_counts.rename_axis('genre').to_frame('movie_count'))
genre_counts.sort_values().plot(kind='barh', figsize=(9, 7))
plt.title('Movies by genre label')
plt.xlabel('Number of movie-genre assignments')
plt.ylabel('Genre label')
plt.tight_layout()
plt.show()

## 10. Tags and external identifiers
Tags can enrich content features. IMDb/TMDB identifiers may help match metadata, but coverage must be checked first.

In [ ]:
print('Tag rows:', len(tags))
print('Unique tags (case-sensitive):', tags['tag'].nunique(dropna=True))
print('Unique users who supplied tags:', tags['userId'].nunique())
print('Unique movies with tags:', tags['movieId'].nunique())
display(tags['tag'].value_counts().head(20).rename_axis('tag').to_frame('count'))

link_coverage = pd.DataFrame([
    {'identifier': 'imdbId', 'non_null_count': int(links['imdbId'].notna().sum()), 'total_rows': len(links)},
    {'identifier': 'tmdbId', 'non_null_count': int(links['tmdbId'].notna().sum()), 'total_rows': len(links)},
])
link_coverage['coverage_percent'] = 100 * link_coverage['non_null_count'] / link_coverage['total_rows']
display(link_coverage)

## 11. Timestamp coverage
Timestamps are Unix seconds in this dataset. Convert them to UTC for summaries; do not interpret them as a user's local time without an explicit timezone decision.

In [ ]:
for name, df in [('ratings', ratings), ('tags', tags)]:
    if 'timestamp' in df.columns and df['timestamp'].notna().any():
        timestamp_utc = pd.to_datetime(df['timestamp'], unit='s', utc=True, errors='coerce')
        print(f'{name}: {timestamp_utc.min()} to {timestamp_utc.max()} (UTC)')
    else:
        print(f'{name}: no usable timestamps found')

## 12. Findings and decisions

After running the notebook, summarize observed results below. Do not record a conclusion unless preceding cells support it.

### Findings
- TODO: Record missing-value and duplicate findings.
- TODO: Record identifier consistency results.
- TODO: Describe the rating distribution and user/movie activity skew.
- TODO: Describe genre and tag coverage.

### Modeling implications
- Content-based features can start with genres and tags; richer descriptions require an additional permitted metadata source.
- Ratings can support a collaborative-filtering baseline, with a carefully defined train/test split to avoid evaluation leakage.
- Cold-start users and items need separate consideration because observed ratings do not cover every possible user–movie pair.

### Limitations
- This research dataset is not necessarily representative of all movie viewers.
- Historical ratings and user-generated tags may not reflect current preferences or general audience opinion.
- Review dataset terms and external metadata licensing before public or commercial deployment.

### Next steps
1. Review generated tables and plots.
2. Replace TODO bullets with evidence-backed findings.
3. Update docs/experiments/01-movielens-audit.md with actual results.
4. Design a separately documented preprocessing pipeline after deciding how data-quality issues should be handled.